
# RUSBoost Predictive Maintenance (Fair Comparison Framework)

This notebook implements the **fair-comparison setup** between:

- Random Forest (earlier paper)
- RUSBoost (follow-up paper)
- XGBoost (New Model Added for comparison)

It ensures:
- Same preprocessing
- Same feature space
- Same validation protocol
- Model-specific training only where appropriate

---

### Methodology Statement

To ensure fair comparison, the two frameworks were evaluated under the same data preprocessing, feature space, and validation protocol. However, model hyperparameters were selected in a model-specific manner, since Random Forest and RUSBoost rely on different learning mechanisms and therefore should not be constrained to identical training settings.


## Step 1 — Data Loading and Preparation

In [66]:
import sys
sys.path.append(".")

In [67]:
import os
print(os.getcwd())

/Users/martinwalterampah/Desktop/Final_Research_Project


In [68]:
import importlib
import Scripts.RUSBoostTrain

importlib.reload(Scripts.RUSBoostTrain)
from Scripts.RUSBoostTrain import train_rusboost

In [69]:

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder, StandardScaler
from Scripts.RUSBoostTrain import train_rusboost

%load_ext autoreload
%autoreload 2

CSV_PATH = "Data_file/ai4i2020.csv"

df = pd.read_csv(CSV_PATH)

print("Dataset shape:", df.shape)
df.head()


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Dataset shape: (10000, 14)


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [70]:
#model, X_test, y_test, y_pred_rus, y_proba_rus, results = train_rusboost("configs/RUSBoost/RUStrain.yaml")
rus_raw, rus_calibrated, X_test, y_test, y_pred_rus, y_proba_rus, results = train_rusboost("configs/RUSBoost/RUStrain.yaml")

Chosen Threshold: 0.01


AttributeError: 'float' object has no attribute 'astype'

In [71]:
print("Chosen threshold:", results["threshold"])
print("PR-AUC:", results["pr_auc"])
print("ROC-AUC:", results["roc_auc"])

Chosen threshold: 0.01
PR-AUC: 0.6306779905553481
ROC-AUC: 0.9556844476921204


In [72]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_rus))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rus, digits=4))

print("\nROC AUC:", roc_auc_score(y_test, y_proba_rus))
print("Average Precision (PR AUC):", average_precision_score(y_test, y_proba_rus))

print("\nChosen Threshold:", results["threshold"])

Confusion Matrix:
[[1753  179]
 [   4   64]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9977    0.9073    0.9504      1932
           1     0.2634    0.9412    0.4116        68

    accuracy                         0.9085      2000
   macro avg     0.6305    0.9243    0.6810      2000
weighted avg     0.9728    0.9085    0.9321      2000


ROC AUC: 0.9556844476921204
Average Precision (PR AUC): 0.6306779905553481

Chosen Threshold: 0.01


In [73]:
# =========================================================
# RUSBoost Experiment Tracking (timestamped run)
# =========================================================

import os
import json
import yaml
from datetime import datetime
import joblib
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, average_precision_score

# --- 1. Timestamp + run dir ---
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
run_dir = os.path.join("experiments_rusboost", f"run_{timestamp}")
os.makedirs(run_dir, exist_ok=True)

print(f"[RUSBoost] Run dir: {run_dir}")

# --- 2. Save model (optional) ---
model_path = os.path.join(run_dir, "rusboost_calibrated.pkl")
#joblib.dump(model, model_path)
joblib.dump(rus_raw, os.path.join(run_dir, "rus_raw.pkl"))
joblib.dump(rus_calibrated, os.path.join(run_dir, "rus_calibrated.pkl"))

print(f"Model saved: {model_path}")

# --- 3. Save config ---
config_src = "configs/RUSBoost/RUStrain.yaml"
config_dst = os.path.join(run_dir, "RUStrain.yaml")

with open(config_src, "r") as f:
    cfg_data = yaml.safe_load(f)

with open(config_dst, "w") as f:
    yaml.dump(cfg_data, f)

print(f"Config saved: {config_dst}")

# --- 4. Save metrics ---
metrics = {
    "model": "RUSBoost",
    "confusion_matrix": confusion_matrix(y_test, y_pred_rus).tolist(),
    "classification_report": classification_report(y_test, y_pred_rus, output_dict=True),
    "roc_auc": roc_auc_score(y_test, y_proba_rus),
    "pr_auc": average_precision_score(y_test, y_proba_rus),
    "threshold": results["threshold"]
}

metrics_path = os.path.join(run_dir, "metrics.json")

with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=4)

print(f"Metrics saved: {metrics_path}")

# --- 5. Create notes file ---
notes_path = os.path.join(run_dir, "notes.txt")

with open(notes_path, "w") as f:
    f.write(
        "RUSBoost Experiment Notes\n"
        "=========================\n\n"
        f"Timestamp: {timestamp}\n\n"
        "Observations:\n- \n\n"
        "Changes Made:\n- \n\n"
        "Next Steps:\n- \n"
    )

print(f"Notes file created: {notes_path}")

[RUSBoost] Run dir: experiments_rusboost/run_2026-04-21_18-02-32
Model saved: experiments_rusboost/run_2026-04-21_18-02-32/rusboost_calibrated.pkl
Config saved: experiments_rusboost/run_2026-04-21_18-02-32/RUStrain.yaml
Metrics saved: experiments_rusboost/run_2026-04-21_18-02-32/metrics.json
Notes file created: experiments_rusboost/run_2026-04-21_18-02-32/notes.txt


In [9]:

selected_uids = {
    "TWF": [2672, 3866, 6341, 8358, 9019],
    "HDF": [3237, 4079, 4174, 4327, 4502],
    "PWF": [464, 1493, 3001, 7537, 8583],
    "OSF": [250, 3020, 5400, 7592, 9660],
}

all_selected_uids = [uid for group in selected_uids.values() for uid in group]

selected_cases = df[df["UDI"].isin(all_selected_uids)].copy().sort_values("UDI")

selected_cases[["UDI", "Type", "Machine failure", "TWF", "HDF", "PWF", "OSF"]]


,UDI,Type,Machine failure,TWF,HDF,PWF,OSF
249,250,L,1,0,0,0,1
463,464,L,1,0,0,1,0
1492,1493,M,1,0,0,1,0
2671,2672,M,1,1,0,0,0
3000,3001,H,1,0,0,1,0
3019,3020,L,1,0,0,0,1
3236,3237,M,1,0,1,0,0
3865,3866,H,1,1,0,0,0
4078,4079,H,1,0,1,0,0
4173,4174,M,1,0,1,0,0
